In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/features/btcusdt_features.parquet"
)

print(df.shape)
df.head()

(29758, 38)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,...,volatility_168h,volume_ma24,volume_ratio,volume_change,bb_upper,bb_lower,bb_width,bb_position,price_vs_ema50,price_vs_ema200
0,2023-01-08 00:00:00,16943.83,16955.00,16911.00,16926.66,6117.02466,1673139599999,1.035801e+08,170975,2837.90539,...,0.001581,4382.346110,1.395833,0.697399,16951.514810,16913.884190,0.002223,0.339506,1.001759,1.006416
1,2023-01-08 01:00:00,16926.66,16936.00,16915.25,16929.82,4392.91927,1673143199999,7.436271e+07,135402,2146.18498,...,0.001579,4387.565009,1.001220,-0.281854,16951.317299,16913.635701,0.002226,0.429501,1.001869,1.006522
2,2023-01-08 02:00:00,16929.82,16934.00,16919.07,16931.18,3389.34917,1673146799999,5.737314e+07,111041,1658.13906,...,0.001579,4354.147813,0.778418,-0.228452,16951.216026,16913.525974,0.002226,0.468400,1.001873,1.006523
3,2023-01-08 03:00:00,16931.46,16944.62,16926.17,16944.02,4396.50361,1673150399999,7.444849e+07,132529,2301.93658,...,0.001577,4369.973383,1.006071,0.297153,16950.407341,16913.927659,0.002153,0.824907,1.002529,1.007197
4,2023-01-08 04:00:00,16944.02,16944.99,16934.40,16934.75,2924.64040,1673153999999,4.953765e+07,99562,1416.47211,...,0.001577,4334.416212,0.674748,-0.334780,16950.654803,16914.179197,0.002154,0.563961,1.001903,1.006565


In [2]:
train_df = df[
    df["open_time"] < "2025-01-01"
].copy()

val_df = df[
    (df["open_time"] >= "2025-01-01") &
    (df["open_time"] < "2025-10-01")
].copy()

test_df = df[
    df["open_time"] >= "2025-10-01"
].copy()

In [3]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 17374
Validation: 6552
Test: 5832


In [4]:
feature_columns = [
    "return",
    "log_return",
    "high_low_spread",
    "open_close_spread",

    "ema20_ema50_ratio",
    "ema50_ema200_ratio",

    "rsi",

    "macd",
    "macd_signal",
    "macd_hist",

    "atr",

    "volatility_24h",
    "volatility_168h",

    "volume_ratio",
    "volume_change",

    "bb_width",
    "bb_position",

    "price_vs_ema50",
    "price_vs_ema200"
]

In [5]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

train_df.loc[:, feature_columns] = (
    scaler.fit_transform(
        train_df[feature_columns]
    )
)

val_df.loc[:, feature_columns] = (
    scaler.transform(
        val_df[feature_columns]
    )
)

test_df.loc[:, feature_columns] = (
    scaler.transform(
        test_df[feature_columns]
    )
)

In [6]:
train_df[feature_columns].describe().T[
    ["mean", "std"]
].head()

,mean,std
return,2.044845e-19,1.000029
log_return,-1.104216e-17,1.000029
high_low_spread,-1.701311e-16,1.000029
open_close_spread,1.083768e-17,1.000029
ema20_ema50_ratio,7.014636e-15,1.000029


In [7]:
joblib.dump(
    scaler,
    "../models/feature_scaler.pkl"
)

print("Scaler saved")

Scaler saved


In [8]:
train_df.to_parquet(
    "../data/processed/train.parquet",
    index=False
)

val_df.to_parquet(
    "../data/processed/val.parquet",
    index=False
)

test_df.to_parquet(
    "../data/processed/test.parquet",
    index=False
)

print("Datasets saved")

Datasets saved
